In [ ]:
# =========================
# 1. Libraries
# =========================
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

# =========================
# 2. Load Data
# =========================
train_df = pd.read_csv('/content/train_patch_features.csv')
test_df = pd.read_csv('/content/test_patch_features.csv')

# =========================
# 3. Prepare Data
# =========================
features = ['GlobalSNR', 'SNR_TL', 'SNR_TR', 'SNR_BL', 'SNR_BR', 'SNR_std', 'SNR_max_min_diff']

X = train_df[features].copy()
y = train_df['label'].astype(int)
X_test = test_df[features].copy()

# Fix inf/nan
X.replace([np.inf, -np.inf], np.nan, inplace=True)
X.fillna(X.mean(), inplace=True)
X_test.replace([np.inf, -np.inf], np.nan, inplace=True)
X_test.fillna(X.mean(), inplace=True)

# =========================
# 4. Scale Features
# =========================
scaler = StandardScaler()
X = scaler.fit_transform(X)
X_test = scaler.transform(X_test)

# =========================
# 5. Train-Validation Split
# =========================
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# =========================
# 6. Train SVM Classifier
# =========================
svm_model = SVC(
    kernel='rbf',
    C=1.0,
    probability=True,  # So we can get probabilities for ROC AUC
    random_state=42
)

svm_model.fit(X_train, y_train)

# =========================
# 7. Evaluate
# =========================
val_preds = svm_model.predict_proba(X_val)[:,1]
val_auc = roc_auc_score(y_val, val_preds)
print(f"✅ Validation AUC (SVM): {val_auc:.5f}")

# =========================
# 8. Predict on Test Set
# =========================
test_preds = svm_model.predict_proba(X_test)[:,1]

# Default threshold 0.5
test_labels = (test_preds > 0.5).astype(int)

# =========================
# 9. Submission
# =========================
submission = pd.DataFrame({
    'id': test_df['id'],
    'label': test_labels
})

submission['id'] = submission['id'].apply(lambda x: f'test_data_v2/{x}')
submission.to_csv('/content/final_patch_snr_svm_submission.csv', index=False)

print("🎯 Final Submission File created: final_patch_snr_svm_submission.csv")


✅ Validation AUC (SVM): 0.76863
🎯 Final Submission File created: final_patch_snr_svm_submission.csv


In [ ]:
!pip install tensorflow


In [ ]:
# ======================
# Phase: Neural Network on Patch-SNR Features
# ======================

import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# === Load Data
train_full = pd.read_csv('/content/train_patch_features.csv')
test_full = pd.read_csv('/content/test_patch_features.csv')

# === Features
features_to_use = ['GlobalSNR', 'SNR_std', 'SNR_max_min_diff']
X = train_full[features_to_use].copy()
X_test = test_full[features_to_use].copy()
y = train_full['label'].astype(int)

# === Clean Data
X.replace([np.inf, -np.inf], np.nan, inplace=True)
X.fillna(X.mean(), inplace=True)
X_test.replace([np.inf, -np.inf], np.nan, inplace=True)
X_test.fillna(X.mean(), inplace=True)

# === Split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# === Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# === Build Model
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# === Train Model
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
model.fit(X_train_scaled, y_train, validation_data=(X_val_scaled, y_val),
          epochs=50, batch_size=64, callbacks=[early_stop], verbose=0)

# === Threshold Optimization
val_preds = model.predict(X_val_scaled).flatten()
best_thresh = 0.5
best_f1 = 0
for thresh in np.linspace(0.3, 0.7, 200):
    preds_binary = (val_preds > thresh).astype(int)
    f1 = f1_score(y_val, preds_binary)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = thresh

print(f"🚀 Best Threshold: {best_thresh:.4f}")
print(f"✅ Best F1 Score: {best_f1:.5f}")

# === Predict on Test
test_preds = model.predict(X_test_scaled).flatten()
final_labels = (test_preds > best_thresh).astype(int)

# === Submission
submission = pd.DataFrame({
    'id': test_full['id'].apply(lambda x: f'test_data_v2/{x}'),
    'label': final_labels
})
submission.to_csv('/content/final_patch_snr_nn_submission.csv', index=False)
print("🎯 Final Submission Created Successfully!")


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
🚀 Best Threshold: 0.3503
✅ Best F1 Score: 0.68836
174/174 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
🎯 Final Submission Created Successfully!


In [ ]:
# ===============================================
# Patch-SNR Based Neural Network Classifier
# ===============================================

# ================
# 1. Libraries
# ================
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ================
# 2. Load Data
# ================
train_full = pd.read_csv('/content/train_patch_features.csv')
test_full = pd.read_csv('/content/test_patch_features.csv')

# ================
# 3. Feature Selection
# ================
# Option 1: Important Patch SNR Features
selected_features = [
   'GlobalSNR', 'SNR_std', 'SNR_max_min_diff'
]

X = train_full[selected_features].copy()
X_test = test_full[selected_features].copy()
y = train_full['label'].astype(int)

# ================
# 4. Clean Data
# ================
X.replace([np.inf, -np.inf], np.nan, inplace=True)
X.fillna(X.mean(), inplace=True)
X_test.replace([np.inf, -np.inf], np.nan, inplace=True)
X_test.fillna(X.mean(), inplace=True)

# ================
# 5. Train-Validation Split
# ================
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ================
# 6. Build Neural Network
# ================
model = keras.Sequential([
    layers.Dense(128, input_shape=(X_train.shape[1],)),
    layers.LeakyReLU(),
    layers.Dropout(0.2),

    layers.Dense(64),
    layers.LeakyReLU(),
    layers.Dropout(0.2),

    layers.Dense(32),
    layers.LeakyReLU(),
    layers.Dropout(0.1),

    layers.Dense(1, activation='sigmoid')
])

optimizer = keras.optimizers.Adam(learning_rate=1e-3)
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['AUC'])

# ================
# 7. Train Model
# ================
early_stopping = keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=128,
    callbacks=[early_stopping, reduce_lr],
    verbose=2
)

# ================
# 8. Evaluate
# ================
val_preds = model.predict(X_val).ravel()
val_auc = roc_auc_score(y_val, val_preds)
print(f"\n✅ Validation AUC: {val_auc:.5f}")

# Find Best Threshold
best_thresh = 0.5
best_f1 = 0
for thresh in np.linspace(0.3, 0.7, 200):
    preds_binary = (val_preds > thresh).astype(int)
    f1 = f1_score(y_val, preds_binary)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = thresh

print(f"🚀 Best Threshold: {best_thresh:.4f}")
print(f"✅ Best F1 Score: {best_f1:.5f}")

# ================
# 9. Predict on Test Set
# ================
test_preds = model.predict(X_test).ravel()
final_labels = (test_preds > best_thresh).astype(int)

submission = pd.DataFrame({
    'id': test_full['id'],
    'label': final_labels
})
submission['id'] = submission['id'].apply(lambda x: f'test_data_v2/{x}')

submission.to_csv('/content/3_patch_snr_nn_submission.csv', index=False)
print("\n🎯 Final Submission Created Successfully!")


Epoch 1/100


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


500/500 - 9s - 17ms/step - AUC: 0.6341 - loss: 0.6720 - val_AUC: 0.6760 - val_loss: 0.6470 - learning_rate: 1.0000e-03
Epoch 2/100
500/500 - 1s - 3ms/step - AUC: 0.6710 - loss: 0.6507 - val_AUC: 0.6768 - val_loss: 0.6459 - learning_rate: 1.0000e-03
Epoch 3/100
500/500 - 1s - 3ms/step - AUC: 0.6768 - loss: 0.6467 - val_AUC: 0.6785 - val_loss: 0.6435 - learning_rate: 1.0000e-03
Epoch 4/100
500/500 - 2s - 4ms/step - AUC: 0.6788 - loss: 0.6451 - val_AUC: 0.6793 - val_loss: 0.6429 - learning_rate: 1.0000e-03
Epoch 5/100
500/500 - 2s - 3ms/step - AUC: 0.6786 - loss: 0.6452 - val_AUC: 0.6796 - val_loss: 0.6435 - learning_rate: 1.0000e-03
Epoch 6/100
500/500 - 1s - 3ms/step - AUC: 0.6804 - loss: 0.6443 - val_AUC: 0.6795 - val_loss: 0.6454 - learning_rate: 1.0000e-03
Epoch 7/100
500/500 - 2s - 5ms/step - AUC: 0.6798 - loss: 0.6442 - val_AUC: 0.6789 - val_loss: 0.6433 - learning_rate: 1.0000e-03
Epoch 8/100
500/500 - 1s - 3ms/step - AUC: 0.6812 - loss: 0.6433 - val_AUC: 0.6798 - val_loss: 0.6429

In [ ]:
# ======================
# Phase: Improved Neural Network on Patch-SNR Features
# ======================

import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, LeakyReLU
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# === Load Data
train_full = pd.read_csv('/content/train_patch_features.csv')
test_full = pd.read_csv('/content/test_patch_features.csv')

# === Features
features_to_use = ['GlobalSNR', 'SNR_std', 'SNR_max_min_diff']

X = train_full[features_to_use].copy()
X_test = test_full[features_to_use].copy()
y = train_full['label'].astype(int)

# === Feature Engineering (Cross Features)
X['SNR_interaction'] = X['GlobalSNR'] * X['SNR_std']
X['SNR_stability'] = X['SNR_std'] / (X['SNR_max_min_diff'] + 1e-5)

X_test['SNR_interaction'] = X_test['GlobalSNR'] * X_test['SNR_std']
X_test['SNR_stability'] = X_test['SNR_std'] / (X_test['SNR_max_min_diff'] + 1e-5)

# === Clean Data
X.replace([np.inf, -np.inf], np.nan, inplace=True)
X.fillna(X.mean(), inplace=True)
X_test.replace([np.inf, -np.inf], np.nan, inplace=True)
X_test.fillna(X.mean(), inplace=True)

# === Split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# === Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# === Build Improved Model
model = Sequential([
    Dense(128, input_shape=(X_train_scaled.shape[1],)),
    BatchNormalization(),
    LeakyReLU(),
    Dropout(0.4),

    Dense(64),
    BatchNormalization(),
    LeakyReLU(),
    Dropout(0.3),

    Dense(32),
    BatchNormalization(),
    LeakyReLU(),
    Dropout(0.2),

    Dense(1, activation='sigmoid')
])

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

# === Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=1)

# === Train
history = model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=100,
    batch_size=128,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

# === Threshold Optimization
val_preds = model.predict(X_val_scaled).flatten()
best_thresh = 0.5
best_f1 = 0
for thresh in np.linspace(0.3, 0.7, 200):
    preds_binary = (val_preds > thresh).astype(int)
    f1 = f1_score(y_val, preds_binary)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = thresh

print(f"\U0001F680 Best Threshold: {best_thresh:.4f}")
print(f"\u2705 Best F1 Score: {best_f1:.5f}")

# === Predict on Test
test_preds = model.predict(X_test_scaled).flatten()
final_labels = (test_preds > best_thresh).astype(int)

# === Submission
submission = pd.DataFrame({
    'id': test_full['id'].apply(lambda x: f'test_data_v2/{x}'),
    'label': final_labels
})
submission.to_csv('/content/final_patch_snr_nn_submission_improved.csv', index=False)
print("\ Final Improved Submission Created Successfully!")

Epoch 1/100


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


500/500 ━━━━━━━━━━━━━━━━━━━━ 12s 9ms/step - accuracy: 0.5989 - loss: 0.7103 - val_accuracy: 0.6341 - val_loss: 0.6452 - learning_rate: 0.0010
Epoch 2/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.6248 - loss: 0.6565 - val_accuracy: 0.6325 - val_loss: 0.6448 - learning_rate: 0.0010
Epoch 3/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.6333 - loss: 0.6489 - val_accuracy: 0.6319 - val_loss: 0.6453 - learning_rate: 0.0010
Epoch 4/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.6384 - loss: 0.6454 - val_accuracy: 0.6325 - val_loss: 0.6443 - learning_rate: 0.0010
Epoch 5/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6415 - loss: 0.6443 - val_accuracy: 0.6317 - val_loss: 0.6444 - learning_rate: 0.0010
Epoch 6/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6362 - loss: 0.6463 - val_accuracy: 0.6345 - val_loss: 0.6442 - learning_rate: 0.0010
Epoch 7/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6345 - loss: 0.6443 - val_

In [ ]:
# ======================
# Neural Network Ensemble on Patch-SNR Features
# ======================

import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import set_random_seed

# === Load Data
train_df = pd.read_csv('/content/train_patch_features.csv')
test_df = pd.read_csv('/content/test_patch_features.csv')

# === Features
features = ['GlobalSNR', 'SNR_std', 'SNR_max_min_diff']
X = train_df[features].copy()
X_test = test_df[features].copy()
y = train_df['label'].astype(int)

# === Clean Data
X.replace([np.inf, -np.inf], np.nan, inplace=True)
X.fillna(X.mean(), inplace=True)
X_test.replace([np.inf, -np.inf], np.nan, inplace=True)
X_test.fillna(X.mean(), inplace=True)

# === Split and Scale
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# === Function to build NN
def build_model(seed=None):
    if seed:
        set_random_seed(seed)
    model = Sequential([
        Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# === Ensemble Training
val_preds_list = []
test_preds_list = []

for i in range(3):  # Ensemble of 3 models
    model = build_model(seed=42 + i)
    early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    model.fit(X_train_scaled, y_train, validation_data=(X_val_scaled, y_val),
              epochs=50, batch_size=64, callbacks=[early_stop], verbose=0)

    val_preds_list.append(model.predict(X_val_scaled).flatten())
    test_preds_list.append(model.predict(X_test_scaled).flatten())

# === Average predictions
val_preds_ensemble = np.mean(val_preds_list, axis=0)
test_preds_ensemble = np.mean(test_preds_list, axis=0)

# === Optimize threshold
best_thresh = 0.5
best_f1 = 0
for thresh in np.linspace(0.3, 0.7, 200):
    f1 = f1_score(y_val, (val_preds_ensemble > thresh).astype(int))
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = thresh

print(f"🚀 Best Threshold: {best_thresh:.4f}")
print(f"✅ Best F1 Score: {best_f1:.5f}")

# === Final Prediction
final_labels = (test_preds_ensemble > best_thresh).astype(int)
submission = pd.DataFrame({
    'id': test_df['id'].apply(lambda x: f'test_data_v2/{x}'),
    'label': final_labels
})
submission.to_csv('/content/final_patch_snr_nn_ensemble_submission.csv', index=False)
print("🎯 Final Ensemble Submission Saved!")


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
174/174 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
174/174 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
174/174 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
🚀 Best Threshold: 0.3281
✅ Best F1 Score: 0.68836
🎯 Final Ensemble Submission Saved!


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

# Load the data
train_df = pd.read_csv("train_patch_features.csv")
test_df = pd.read_csv("test_patch_features.csv")

# Features and target
features = ['GlobalSNR', 'SNR_std', 'SNR_max_min_diff']
X = train_df[features].copy()
y = train_df['label'].astype(int)
X_test = test_df[features].copy()

# Clean data
X.replace([np.inf, -np.inf], np.nan, inplace=True)
X.fillna(X.mean(), inplace=True)
X_test.replace([np.inf, -np.inf], np.nan, inplace=True)
X_test.fillna(X.mean(), inplace=True)

# Split and scale
X_train, X_val, y_train, y_val = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Build improved neural net
model = Sequential([
    Dense(256, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    BatchNormalization(),
    Dropout(0.4),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
model.fit(X_train_scaled, y_train, validation_data=(X_val_scaled, y_val),
          epochs=100, batch_size=64, callbacks=[early_stop], verbose=1)

# Threshold tuning
val_preds = model.predict(X_val_scaled).flatten()
best_thresh, best_f1 = 0.5, 0
for thresh in np.linspace(0.3, 0.7, 300):
    preds = (val_preds > thresh).astype(int)
    f1 = f1_score(y_val, preds)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = thresh

print(f"✅ Best F1 Score: {best_f1:.5f} at Threshold: {best_thresh:.4f}")

# Final prediction
test_preds = model.predict(X_test_scaled).flatten()
final_labels = (test_preds > best_thresh).astype(int)

# Submission
submission = pd.DataFrame({
    'id': test_df['id'].apply(lambda x: f'test_data_v2/{x}'),
    'label': final_labels
})
submission.to_csv("final_patch_snr_nn_submission_last_push.csv", index=False)
print("✅ Final submission saved!")

Epoch 1/100


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1000/1000 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - accuracy: 0.6064 - loss: 0.6989 - val_accuracy: 0.6334 - val_loss: 0.6474
Epoch 2/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6339 - loss: 0.6482 - val_accuracy: 0.6350 - val_loss: 0.6444
Epoch 3/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.6387 - loss: 0.6446 - val_accuracy: 0.6349 - val_loss: 0.6445
Epoch 4/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6388 - loss: 0.6429 - val_accuracy: 0.6354 - val_loss: 0.6446
Epoch 5/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6379 - loss: 0.6433 - val_accuracy: 0.6340 - val_loss: 0.6445
Epoch 6/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6396 - loss: 0.6433 - val_accuracy: 0.6341 - val_loss: 0.6443
Epoch 7/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6380 - loss: 0.6424 - val_accuracy: 0.6332 - val_loss: 0.6449
Epoch 8/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.6401 - loss: 0.6426 - val

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

# Load the data
train_df = pd.read_csv("train_patch_features.csv")
test_df = pd.read_csv("test_patch_features.csv")

# Features and target
features = ['GlobalSNR']
X = train_df[features].copy()
y = train_df['label'].astype(int)
X_test = test_df[features].copy()

# Clean data
X.replace([np.inf, -np.inf], np.nan, inplace=True)
X.fillna(X.mean(), inplace=True)
X_test.replace([np.inf, -np.inf], np.nan, inplace=True)
X_test.fillna(X.mean(), inplace=True)

# Split and scale
X_train, X_val, y_train, y_val = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Build improved neural net
model = Sequential([
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
model.fit(X_train_scaled, y_train, validation_data=(X_val_scaled, y_val),
          epochs=100, batch_size=64, callbacks=[early_stop], verbose=1)

# Threshold tuning
val_preds = model.predict(X_val_scaled).flatten()
best_thresh, best_f1 = 0.5, 0
for thresh in np.linspace(0.3, 0.7, 300):
    preds = (val_preds > thresh).astype(int)
    f1 = f1_score(y_val, preds)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = thresh

print(f"✅ Best F1 Score: {best_f1:.5f} at Threshold: {best_thresh:.4f}")

# Final prediction
test_preds = model.predict(X_test_scaled).flatten()
final_labels = (test_preds > best_thresh).astype(int)

# Submission
submission = pd.DataFrame({
    'id': test_df['id'].apply(lambda x: f'test_data_v2/{x}'),
    'label': final_labels
})
submission.to_csv("final_patch_snr_nn_submission_last_push.csv", index=False)
print("✅ Final submission saved!")

Epoch 1/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - accuracy: 0.6139 - loss: 0.6824 - val_accuracy: 0.6301 - val_loss: 0.6509
Epoch 2/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.6357 - loss: 0.6469 - val_accuracy: 0.6327 - val_loss: 0.6476
Epoch 3/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.6377 - loss: 0.6456 - val_accuracy: 0.6322 - val_loss: 0.6457
Epoch 4/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6394 - loss: 0.6439 - val_accuracy: 0.6317 - val_loss: 0.6465
Epoch 5/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6400 - loss: 0.6433 - val_accuracy: 0.6321 - val_loss: 0.6463
Epoch 6/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6407 - loss: 0.6431 - val_accuracy: 0.6320 - val_loss: 0.6466
Epoch 7/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6399 - loss: 0.6423 - val_accuracy: 0.6328 - val_loss: 0.6459
Epoch 8/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.6401 - loss: 